# 08 — Evidence-Grounded Biological Reasoning for All Clusters

This notebook applies the validated Cluster 5 workflow independently to every annotated PBMC3k cluster detected in Phase 6.

For each cluster it loads only that cluster's Phase 6 observations and Phase 7 evidence, exports the unchanged reasoning prompt, optionally loads a manually supplied JSON response, applies the unchanged validators, and generates the unchanged biological reports.

No API or internet access is used. Missing or invalid responses do not prevent other clusters from being processed.

## 1. Imports and project paths

Cluster-specific outputs are written beneath `results/phase8/cluster_X/`. Existing flat Cluster 5 artifacts are retained as regression baselines.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import JSON, Markdown, display
from jsonschema import Draft202012Validator

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "results" / "phase6").is_dir()
            and (candidate / "results" / "phase7").is_dir()
            and (candidate / "results" / "tables").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the PBMC3k project root.")

PROJECT = find_project_root(Path.cwd().resolve())
PHASE6 = PROJECT / "results" / "phase6"
PHASE7 = PROJECT / "results" / "phase7"
PHASE8 = PROJECT / "results" / "phase8"
TABLES = PROJECT / "results" / "tables"
PHASE8.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT}")
print(f"Phase 8 output: {PHASE8}")
print("Execution mode: fully offline, multi-cluster prompt generation and validation")

Project root: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k
Phase 8 output: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k/results/phase8
Execution mode: fully offline, multi-cluster prompt generation and validation


## 2. Load the existing Phase 6 and Phase 7 evidence

The same saved marker, annotation, classifier, literature-summary, reference, and cluster-report files used by the validated Cluster 5 workflow are loaded once. Cluster filtering occurs before prompt construction.

In [2]:
input_paths = {
    "selected_markers": PHASE6 / "selected_marker_genes.csv",
    "pilot_summary": PHASE6 / "pilot_cluster_summary.md",
    "literature_summary": PHASE7 / "literature_summary.csv",
    "cluster_report": PHASE7 / "cluster_reference_report.md",
    "verified_references": PHASE7 / "references.csv",
    "annotations": TABLES / "leiden_9_cell_type_annotations.csv",
    "model_comparison": TABLES / "classification_model_comparison.csv",
    "model_per_class": TABLES / "classification_best_model_per_class.csv",
}
missing_inputs = [str(path) for path in input_paths.values() if not path.exists()]
if missing_inputs:
    raise FileNotFoundError("Missing required inputs:\n- " + "\n- ".join(missing_inputs))

selected_all = pd.read_csv(input_paths["selected_markers"], dtype={"cluster": str})
literature = pd.read_csv(input_paths["literature_summary"], dtype={"cluster": str})
references = pd.read_csv(input_paths["verified_references"], dtype={"PMID": str})
annotations = pd.read_csv(input_paths["annotations"], dtype={"leiden": str})
model_comparison = pd.read_csv(input_paths["model_comparison"], index_col=0)
model_per_class = pd.read_csv(input_paths["model_per_class"], index_col=0)
all_cluster_report_text = input_paths["cluster_report"].read_text(encoding="utf-8")

required_columns = {
    "selected markers": (selected_all, {"cluster", "cell_type", "gene", "representative_rank", "avg_log2FC", "adjusted_p_value", "pct_in", "pct_out", "marker_score"}),
    "literature summary": (literature, {"gene", "cluster", "evidence_grade", "immune_function", "biological_role", "grade_explanation"}),
    "references": (references, {"gene", "PMID", "title", "summary", "evidence_grade"}),
}
for label, (frame, required) in required_columns.items():
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{label} is missing required columns: {sorted(missing)}")

print("Loaded inputs:")
for label, input_path in input_paths.items():
    print(f"- {label}: {input_path.relative_to(PROJECT)}")

Loaded inputs:
- selected_markers: results/phase6/selected_marker_genes.csv
- pilot_summary: results/phase6/pilot_cluster_summary.md
- literature_summary: results/phase7/literature_summary.csv
- cluster_report: results/phase7/cluster_reference_report.md
- verified_references: results/phase7/references.csv
- annotations: results/tables/leiden_9_cell_type_annotations.csv
- model_comparison: results/tables/classification_model_comparison.csv
- model_per_class: results/tables/classification_best_model_per_class.csv


## 3. Discover clusters and isolate cluster reference reports

Clusters come directly from Phase 6. The combined Phase 7 report is split at its cluster headings so no prompt receives another cluster's report.

In [3]:
def cluster_sort_key(value):
    text = str(value)
    return (int(text), text) if text.isdigit() else (10**9, text)

all_clusters = sorted(selected_all["cluster"].dropna().astype(str).unique(), key=cluster_sort_key)
phase7_clusters = set(literature["cluster"].dropna().astype(str))
annotation_clusters = set(annotations["leiden"].dropna().astype(str))

if set(all_clusters) != phase7_clusters:
    raise ValueError(
        f"Phase 6/7 cluster coverage differs. Phase 6={all_clusters}; "
        f"Phase 7={sorted(phase7_clusters, key=cluster_sort_key)}"
    )
if set(all_clusters) != annotation_clusters:
    raise ValueError("Reviewed annotation coverage differs from Phase 6.")

report_heading = re.compile(
    r"^# Literature Reference Report — Cluster ([^:]+):.*$",
    flags=re.MULTILINE,
)
report_matches = list(report_heading.finditer(all_cluster_report_text))
cluster_report_sections = {}
for index, match in enumerate(report_matches):
    cluster = match.group(1).strip()
    end = report_matches[index + 1].start() if index + 1 < len(report_matches) else len(all_cluster_report_text)
    section = all_cluster_report_text[match.start():end]
    if cluster == "5":
        section = section.split(
            "\n\n---\n\n# Additional Cluster Literature Reference Reports", 1
        )[0]
    section = re.sub(r"\n+---\n+\Z", "\n", section)
    cluster_report_sections[cluster] = section.rstrip() + "\n"

if set(cluster_report_sections) != set(all_clusters):
    raise ValueError(
        "Cluster report coverage differs from Phase 6: "
        f"{sorted(cluster_report_sections, key=cluster_sort_key)}"
    )

cluster_inventory = (
    selected_all.groupby(["cluster", "cell_type"], as_index=False)
    .agg(representative_genes=("gene", "size"))
    .sort_values("cluster", key=lambda values: values.map(cluster_sort_key))
)
display(cluster_inventory)
print(f"Discovered {len(all_clusters)} clusters: {', '.join(all_clusters)}")

,cluster,cell_type,representative_genes
0,0,Cytotoxic CD8 T cells,10
1,1,B cells,10
2,2,IL7R+ memory/helper T cells,10
3,3,Classical monocytes,10
4,4,CD16+ non-classical monocytes,10
5,5,NK cells,10
6,6,Activated/transitional T cells,10
7,7,Naive/resting T cells,10
8,8,Platelets,10


Discovered 9 clusters: 0, 1, 2, 3, 4, 5, 6, 7, 8


## 4. Exact validated JSON contract

This is the unchanged strict JSON Schema from the validated Cluster 5 implementation.

In [4]:
def strict_object(properties, required=None):
    return {
        "type": "object",
        "properties": properties,
        "required": list(properties) if required is None else required,
        "additionalProperties": False,
    }

string_array = {"type": "array", "items": {"type": "string"}}
confidence = {"type": "string", "enum": ["High", "Moderate", "Low"]}

OUTPUT_SCHEMA = strict_object({
    "cluster_id": {"type": "string"},
    "proposed_cell_type": {"type": "string"},
    "annotation_assessment": strict_object({
        "support_level": {
            "type": "string",
            "enum": ["strongly supported", "partially supported", "weakly supported"],
        },
        "supporting_genes": string_array,
        "supporting_evidence": {"type": "string"},
        "alternative_interpretations": string_array,
        "additional_evidence_needed": string_array,
    }),
    "strongest_dataset_observations": {
        "type": "array",
        "minItems": 1,
        "items": strict_object({
            "gene": {"type": "string"},
            "observation": {"type": "string"},
            "importance": {"type": "string"},
        }),
    },
    "functional_modules": {
        "type": "array",
        "minItems": 1,
        "items": strict_object({
            "module_name": {"type": "string"},
            "genes": string_array,
            "dataset_support": {"type": "string"},
            "literature_support": {"type": "string"},
            "biological_inference": {"type": "string"},
            "confidence": confidence,
            "confidence_reason": {"type": "string"},
            "reference_ids": {
                "type": "array",
                "items": {"type": "string", "pattern": r"^PMID:\d+$"},
            },
        }),
    },
    "coordinated_biological_program": {"type": "string"},
    "supported_conclusions": string_array,
    "reasonable_inferences": string_array,
    "weak_or_uncertain_interpretations": string_array,
    "contradictory_evidence": string_array,
    "limitations": string_array,
    "overall_confidence": confidence,
    "overall_confidence_reason": {"type": "string"},
    "plain_language_explanation": {"type": "string"},
})

schema_errors = list(Draft202012Validator.check_schema(OUTPUT_SCHEMA) or [])
print("JSON Schema is valid.")

JSON Schema is valid.


## 5. Exact validated reasoning prompt

The prompt wording below is unchanged. Each placeholder is filled independently with one cluster's evidence.

In [5]:
PROMPT_TEMPLATE = """You are an evidence-grounded biological reasoning assistant specializing in
single-cell RNA sequencing and immune-cell biology.

Your task is to interpret one cell cluster using only the dataset observations
and literature evidence provided below.

Do not use unsupported outside knowledge as evidence. You may connect facts
logically, but every inference must be clearly labeled as an inference.

==================================================
CLUSTER INFORMATION
==================================================

Cluster ID:
{cluster_id}

Current cell-type annotation:
{cell_type_annotation}

Classification-model prediction:
{model_prediction}

Classification confidence, if available:
{model_confidence}

==================================================
DATASET OBSERVATIONS
==================================================

Representative marker genes:
{representative_genes}

Marker-gene statistics:
{marker_gene_table}

For each gene, the table may include:
- average log2 fold change
- adjusted p-value
- percentage expressed inside the cluster
- percentage expressed outside the cluster
- marker score
- rank

==================================================
VERIFIED LITERATURE EVIDENCE
==================================================

Gene evidence summaries:
{gene_evidence_summaries}

Cluster-level literature summary:
{cluster_literature_summary}

Evidence grades:
{evidence_grades}

Verified references:
{verified_references}

==================================================
REASONING INSTRUCTIONS
==================================================

1. Begin with the dataset evidence. Identify the strongest observations based
   on marker specificity, expression prevalence, fold change, statistical
   significance, and marker score.

2. Group the representative genes into functional biological modules. A module
   should contain genes that contribute to the same biological process, cell
   identity, pathway, regulatory function, or communication mechanism.

3. For each functional module:
   - list the relevant genes
   - describe the shared biological function
   - identify which statements are directly supported by the supplied literature
   - explain how the dataset observations support the module
   - assign a confidence level of High, Moderate, or Low
   - explain the reason for that confidence level

4. Evaluate the proposed cell-type annotation:
   - state whether it is strongly supported, partially supported, or weakly supported
   - identify the genes and literature evidence that support it
   - identify plausible alternative cell types or cell states
   - explain what additional evidence would help distinguish the alternatives

5. Produce an overall biological interpretation of the cluster. Explain how the
   functional modules work together as a coordinated biological program. Do not
   merely summarize each gene separately.

6. Clearly distinguish among:
   - DATASET OBSERVATION: directly measured in this analysis
   - LITERATURE-SUPPORTED FACT: reported in the supplied references
   - BIOLOGICAL INFERENCE: a reasoned connection between the dataset and literature
   - UNKNOWN OR UNCERTAIN: not sufficiently supported by the supplied evidence

7. Treat disease-related papers only as research contexts in which a gene or
   pathway was studied. Gene expression in this dataset must not be interpreted
   as evidence that the donor had any disease.

8. Do not:
   - diagnose a disease
   - infer the donor's identity, health status, age, sex, ethnicity, or medical history
   - claim that a marker gene is unique to one cell type unless the supplied evidence
     explicitly establishes uniqueness
   - invent biological mechanisms
   - invent citations, PMIDs, DOIs, numerical values, or experimental results
   - treat correlation as causation
   - claim pathway activation solely because one associated gene is expressed
   - hide contradictory or weak evidence

9. If the evidence is incomplete, explicitly say:
   "The supplied evidence is insufficient to determine this."

10. Cite supplied sources using their PMID or reference identifier. Only cite
    references included in the verified reference list.

==================================================
REQUIRED OUTPUT
==================================================

Return valid JSON using exactly this structure:

{
  "cluster_id": "",
  "proposed_cell_type": "",
  "annotation_assessment": {
    "support_level": "strongly supported | partially supported | weakly supported",
    "supporting_genes": [],
    "supporting_evidence": "",
    "alternative_interpretations": [],
    "additional_evidence_needed": []
  },
  "strongest_dataset_observations": [
    {
      "gene": "",
      "observation": "",
      "importance": ""
    }
  ],
  "functional_modules": [
    {
      "module_name": "",
      "genes": [],
      "dataset_support": "",
      "literature_support": "",
      "biological_inference": "",
      "confidence": "High | Moderate | Low",
      "confidence_reason": "",
      "reference_ids": []
    }
  ],
  "coordinated_biological_program": "",
  "supported_conclusions": [],
  "reasonable_inferences": [],
  "weak_or_uncertain_interpretations": [],
  "contradictory_evidence": [],
  "limitations": [],
  "overall_confidence": "High | Moderate | Low",
  "overall_confidence_reason": "",
  "plain_language_explanation": ""
}

Before returning the JSON, silently verify that:
- every citation exists in the supplied references
- every gene mentioned appears in the supplied data
- observations and inferences are clearly separated
- no diagnosis or donor inference appears
- the output contains valid JSON
"""


## 6. Exact validated response validators

The validator implementation below is unchanged from the validated Cluster 5 notebook. The cluster loop supplies its existing global inputs separately for each response.

In [6]:
def iter_strings(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for child in value.values():
            yield from iter_strings(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_strings(child)

def validate_analysis(result):
    schema_errors = list(response_load_errors)
    gene_errors = []
    citation_errors = []
    safety_errors = []
    warnings = []

    if not isinstance(result, dict):
        if not schema_errors:
            schema_errors.append("The response must be a JSON object.")
        return {
            "valid": False,
            "checks": {
                "schema_passed": False,
                "genes_validated": False,
                "citations_validated": False,
                "safety_passed": False,
            },
            "errors": {
                "schema": schema_errors,
                "genes": gene_errors,
                "citations": citation_errors,
                "safety": safety_errors,
            },
            "warnings": warnings,
        }

    schema_validator = Draft202012Validator(OUTPUT_SCHEMA)
    for error in sorted(schema_validator.iter_errors(result), key=lambda e: list(e.path)):
        location = ".".join(map(str, error.path)) or "<root>"
        schema_errors.append(f"Schema at {location}: {error.message}")

    if result.get("cluster_id") != pilot_cluster:
        schema_errors.append(f"cluster_id must be {pilot_cluster!r}.")

    assessment = result.get("annotation_assessment", {})
    supporting_genes = assessment.get("supporting_genes", []) if isinstance(assessment, dict) else []
    explicit_genes = {gene for gene in supporting_genes if isinstance(gene, str) and gene}
    observations = result.get("strongest_dataset_observations", [])
    if isinstance(observations, list):
        explicit_genes.update(
            row.get("gene") for row in observations
            if isinstance(row, dict) and isinstance(row.get("gene"), str) and row.get("gene")
        )
    modules = result.get("functional_modules", [])
    if not isinstance(modules, list):
        modules = []
    for module in modules:
        genes = module.get("genes", []) if isinstance(module, dict) else []
        if isinstance(genes, list):
            explicit_genes.update(gene for gene in genes if isinstance(gene, str) and gene)
    unsupported_explicit_genes = explicit_genes.difference(marker_genes)
    if unsupported_explicit_genes:
        gene_errors.append(
            "Structured gene fields contain genes outside the supplied pilot data: "
            f"{sorted(unsupported_explicit_genes)}"
        )

    cited_ids = {
        citation
        for module in modules
        if isinstance(module, dict)
        for citation in module.get("reference_ids", []) if isinstance(citation, str)
    }
    unsupported_citations = cited_ids.difference(valid_reference_ids)
    if unsupported_citations:
        citation_errors.append(f"Unsupported reference IDs: {sorted(unsupported_citations)}")

    all_text = "\n".join(iter_strings(result))
    prose_pmids = {
        "PMID:" + match
        for match in re.findall(r"PMID:?\s*(\d+)", all_text, flags=re.IGNORECASE)
    }
    unsupported_prose_pmids = prose_pmids.difference(valid_reference_ids)
    if unsupported_prose_pmids:
        citation_errors.append(
            f"Unsupported PMID citations in prose: {sorted(unsupported_prose_pmids)}"
        )

    # Preserve the existing safeguard against known project genes absent from this pilot packet.
    project_gene_universe = set(selected_all["gene"].astype(str))
    mentioned_known_genes = {
        gene for gene in project_gene_universe
        if re.search(rf"(?<![A-Za-z0-9]){re.escape(gene)}(?![A-Za-z0-9])", all_text)
    }
    leaked_known_genes = mentioned_known_genes.difference(marker_genes)
    if leaked_known_genes:
        gene_errors.append(
            "Text mentions project marker genes not supplied for this cluster: "
            f"{sorted(leaked_known_genes)}"
        )

    risky_claim_patterns = {
        "diagnosis claim": r"\b(diagnos(?:e|ed|is)|the donor (?:has|had|suffers from))\b",
        "donor identity inference": r"\bthe donor(?:'s| is| was).{0,40}\b(identity|name)\b",
        "donor demographic inference": r"\bthe donor(?:'s| is| was).{0,40}\b(age|sex|gender|ethnicity|medical history)\b",
        "unsupported health claim": r"\bthe donor(?:'s| is| was| has| had).{0,50}\b(healthy|unhealthy|disease|condition|infection|cancer|illness)\b",
        "causal overclaim": r"\b(expression of|expressing)\b.{0,50}\b(causes|proves|demonstrates that the donor)\b",
    }
    for label, pattern in risky_claim_patterns.items():
        if re.search(pattern, all_text, flags=re.IGNORECASE | re.DOTALL):
            safety_errors.append(f"Potential {label} detected; review the output.")

    if not cited_ids:
        warnings.append("No module reference_ids were supplied; literature-supported claims may be under-cited.")
    if "DATASET OBSERVATION" not in all_text:
        warnings.append("The exact label 'DATASET OBSERVATION' does not appear in the result.")
    if "BIOLOGICAL INFERENCE" not in all_text:
        warnings.append("The exact label 'BIOLOGICAL INFERENCE' does not appear in the result.")

    checks = {
        "schema_passed": not schema_errors,
        "genes_validated": not gene_errors,
        "citations_validated": not citation_errors,
        "safety_passed": not safety_errors,
    }
    return {
        "valid": all(checks.values()),
        "checks": checks,
        "errors": {
            "schema": schema_errors,
            "genes": gene_errors,
            "citations": citation_errors,
            "safety": safety_errors,
        },
        "warnings": warnings,
    }


## 7. Exact validated report builder

The Markdown report-generation functions below are unchanged from the validated Cluster 5 notebook.

In [7]:
def bullet_list(items):
    return "\n".join(f"- {item}" for item in items) if items else "- None supplied."

def build_biological_report(result):
    assessment = result["annotation_assessment"]
    observation_lines = [
        f"- **{item['gene']}** — {item['observation']} Importance: {item['importance']}"
        for item in result["strongest_dataset_observations"]
    ]
    module_sections = []
    for module in result["functional_modules"]:
        references_text = ", ".join(module["reference_ids"]) or "None supplied"
        module_sections.append(
            f"### {module['module_name']}\n\n"
            f"- **Genes:** {', '.join(module['genes'])}\n"
            f"- **Confidence:** {module['confidence']} — {module['confidence_reason']}\n"
            f"- **Dataset support:** {module['dataset_support']}\n"
            f"- **Literature support:** {module['literature_support']}\n"
            f"- **Biological inference:** {module['biological_inference']}\n"
            f"- **Verified references:** {references_text}"
        )

    return f"""# Biological Interpretation Report — Cluster {result['cluster_id']}

## Annotation assessment

- **Proposed cell type:** {result['proposed_cell_type']}
- **Support level:** {assessment['support_level']}
- **Supporting genes:** {', '.join(assessment['supporting_genes'])}
- **Supporting evidence:** {assessment['supporting_evidence']}

### Alternative interpretations

{bullet_list(assessment['alternative_interpretations'])}

### Additional evidence needed

{bullet_list(assessment['additional_evidence_needed'])}

## Strongest dataset observations

{chr(10).join(observation_lines)}

## Functional modules

{chr(10).join(module_sections)}

## Coordinated biological program

{result['coordinated_biological_program']}

## Supported conclusions

{bullet_list(result['supported_conclusions'])}

## Reasonable inferences

{bullet_list(result['reasonable_inferences'])}

## Weak or uncertain interpretations

{bullet_list(result['weak_or_uncertain_interpretations'])}

## Contradictory evidence

{bullet_list(result['contradictory_evidence'])}

## Limitations

{bullet_list(result['limitations'])}

## Overall confidence

**{result['overall_confidence']}** — {result['overall_confidence_reason']}

## Plain-language explanation

{result['plain_language_explanation']}
"""


## 8. Generate prompts, load responses, validate, and report

Each cluster is processed inside its own exception boundary. A missing response is marked `SKIPPED`; a failed response is recorded as `FAIL`; neither condition interrupts another cluster.

In [8]:
PASTED_RESPONSE_JSON_BY_CLUSTER = {
    # Optional alternative to a response file, for example:
    # "0": r"""{"cluster_id": "0", ...}""",
}

master_rows = []
execution_rows = []
generated_prompt_paths = []

for pilot_cluster in all_clusters:
    cluster_output = PHASE8 / f"cluster_{pilot_cluster}"
    cluster_output.mkdir(parents=True, exist_ok=True)
    prompt_path = cluster_output / "cluster_reasoning_prompt.md"
    response_path = cluster_output / "cluster_reasoning_response.json"
    reasoning_path = cluster_output / "cluster_reasoning.json"
    biological_report_path = cluster_output / "biological_interpretation_report.md"
    plain_summary_path = cluster_output / "plain_language_summary.md"
    validation_path = cluster_output / "reasoning_validation_report.json"
    validation_summary_path = cluster_output / "reasoning_validation_summary.json"
    input_manifest_path = cluster_output / "input_manifest.json"

    try:
        pilot_clusters = [pilot_cluster]
        pilot_markers = (
            selected_all[selected_all["cluster"].eq(pilot_cluster)]
            .sort_values("representative_rank")
            .reset_index(drop=True)
        )
        pilot_literature = (
            literature[literature["cluster"].eq(pilot_cluster)]
            .sort_values("representative_rank")
            .reset_index(drop=True)
        )
        pilot_genes = pilot_markers["gene"].astype(str).tolist()
        marker_genes = set(pilot_genes)

        annotation_row = annotations[annotations["leiden"].eq(pilot_cluster)]
        if len(annotation_row) != 1:
            raise ValueError(
                f"Expected one reviewed annotation for cluster {pilot_cluster}; "
                f"found {len(annotation_row)}"
            )
        cell_type_annotation = str(annotation_row.iloc[0]["cell_type"])
        annotation_confidence = str(annotation_row.iloc[0]["confidence"])

        if set(pilot_literature["gene"]) != marker_genes:
            raise ValueError(
                f"Cluster {pilot_cluster}: Phase 6 marker genes and Phase 7 "
                "literature genes do not match."
            )
        if (
            pilot_markers["cell_type"].nunique() != 1
            or pilot_markers["cell_type"].iloc[0] != cell_type_annotation
        ):
            raise ValueError(
                f"Cluster {pilot_cluster}: Phase 6 and reviewed annotations disagree."
            )

        best_model_row = model_comparison.sort_values("rank").iloc[0]
        best_model_name = str(best_model_row["model"])
        if cell_type_annotation in model_per_class.index:
            class_metrics = model_per_class.loc[cell_type_annotation]
            model_metric_context = (
                f"Held-out {cell_type_annotation} metrics for {best_model_name}: "
                f"precision={float(class_metrics['precision']):.3f}, "
                f"recall={float(class_metrics['recall']):.3f}, "
                f"F1={float(class_metrics['f1-score']):.3f}, "
                f"support={int(float(class_metrics['support']))}. "
                "These are evaluation metrics against expression-derived labels, not a cluster-level probability."
            )
        else:
            model_metric_context = (
                f"No held-out per-class row was exported for {cell_type_annotation}."
            )
        model_prediction = (
            "Not available: Notebook 05 did not export a cluster-level prediction. "
            f"The selected cell-level model was {best_model_name}."
        )
        model_confidence = (
            "Not available as a cluster-level probability. " + model_metric_context
        )

        marker_columns = [
            "representative_rank", "gene", "avg_log2FC", "adjusted_p_value",
            "pct_in", "pct_out", "specificity_delta", "marker_score", "selection_tier",
        ]
        marker_gene_table = pilot_markers[marker_columns].copy()
        marker_gene_table["adjusted_p_value"] = marker_gene_table[
            "adjusted_p_value"
        ].map(lambda x: f"{float(x):.6e}")
        for column in [
            "avg_log2FC", "pct_in", "pct_out", "specificity_delta", "marker_score"
        ]:
            marker_gene_table[column] = marker_gene_table[column].map(
                lambda x: round(float(x), 6)
            )

        summary_columns = [
            "gene", "official_gene_name", "immune_function", "immune_cell_contexts",
            "biological_role", "function_tags", "pathway_tags", "grade_explanation",
            "plain_language_note",
        ]
        gene_evidence_summaries = pilot_literature[summary_columns].fillna(
            ""
        ).to_dict(orient="records")
        evidence_grades = pilot_literature[
            ["gene", "evidence_grade", "publication_count", "grade_explanation"]
        ].fillna("").to_dict(orient="records")

        verified_references = references[references["gene"].isin(pilot_genes)].copy()
        verified_references["reference_id"] = (
            "PMID:" + verified_references["PMID"].astype(str)
        )
        reference_columns = [
            "reference_id", "gene", "title", "journal", "year", "DOI",
            "evidence_grade", "study_type", "summary", "evidence_categories",
        ]
        verified_reference_records = verified_references[
            reference_columns
        ].fillna("").to_dict(orient="records")
        valid_reference_ids = set(verified_references["reference_id"])
        cluster_literature_summary = cluster_report_sections[pilot_cluster]

        replacements = {
            "{cluster_id}": pilot_cluster,
            "{cell_type_annotation}": cell_type_annotation,
            "{model_prediction}": model_prediction,
            "{model_confidence}": model_confidence,
            "{representative_genes}": ", ".join(pilot_genes),
            "{marker_gene_table}": marker_gene_table.to_csv(index=False),
            "{gene_evidence_summaries}": json.dumps(
                gene_evidence_summaries, indent=2, ensure_ascii=False
            ),
            "{cluster_literature_summary}": cluster_literature_summary,
            "{evidence_grades}": json.dumps(
                evidence_grades, indent=2, ensure_ascii=False
            ),
            "{verified_references}": json.dumps(
                verified_reference_records, indent=2, ensure_ascii=False
            ),
        }
        assembled_prompt = PROMPT_TEMPLATE
        for placeholder, value in replacements.items():
            assembled_prompt = assembled_prompt.replace(placeholder, str(value))
        unfilled = re.findall(r"\{[a-z_]+\}", assembled_prompt)
        if unfilled:
            raise ValueError(
                f"Cluster {pilot_cluster}: unfilled prompt placeholders "
                f"{sorted(set(unfilled))}"
            )

        reference_genes = set(verified_references["gene"].astype(str))
        literature_genes = set(pilot_literature["gene"].astype(str))
        preflight_checks = {
            "one pilot cluster": len(pilot_clusters) == 1,
            "marker genes equal literature genes": marker_genes == literature_genes,
            "every evidenced pilot gene has a verified reference": set(
                pilot_literature.loc[
                    pilot_literature["publication_count"].gt(0), "gene"
                ]
            ).issubset(reference_genes),
            "zero-reference genes explicitly use grade E": (
                pilot_literature.loc[
                    pilot_literature["publication_count"].eq(0), "evidence_grade"
                ].eq("E").all()
            ),
            "all PMIDs are non-empty numeric strings": (
                verified_references["PMID"].astype(str).str.fullmatch(r"\d+").all()
            ),
            "all evidence grades are present": pilot_literature[
                "evidence_grade"
            ].notna().all(),
            "no prompt placeholders remain": not unfilled,
            "cluster annotation is present": bool(cell_type_annotation.strip()),
            "cluster reference report is isolated": (
                cluster_literature_summary.count(
                    "# Literature Reference Report — Cluster "
                ) == 1
            ),
        }
        failed_preflight = [
            name for name, passed in preflight_checks.items() if not passed
        ]
        if failed_preflight:
            raise ValueError(
                f"Cluster {pilot_cluster} preflight failed: {failed_preflight}"
            )

        prompt_path.write_text(assembled_prompt, encoding="utf-8")
        generated_prompt_paths.append(prompt_path)
        input_manifest = {
            "cluster_id": pilot_cluster,
            "cell_type_annotation": cell_type_annotation,
            "annotation_confidence": annotation_confidence,
            "representative_genes": pilot_genes,
            "model_prediction": model_prediction,
            "model_confidence": model_confidence,
            "verified_reference_ids": sorted(valid_reference_ids),
            "source_files": {
                key: str(source_path.relative_to(PROJECT))
                for key, source_path in input_paths.items()
            },
        }
        input_manifest_path.write_text(
            json.dumps(input_manifest, indent=2), encoding="utf-8"
        )

        response_source = None
        response_text = None
        response_load_errors = []
        analysis = None
        if response_path.exists():
            response_source = str(response_path.relative_to(PROJECT))
            response_text = response_path.read_text(encoding="utf-8")
        elif str(PASTED_RESPONSE_JSON_BY_CLUSTER.get(pilot_cluster, "")).strip():
            response_source = f"PASTED_RESPONSE_JSON_BY_CLUSTER[{pilot_cluster!r}]"
            response_text = str(
                PASTED_RESPONSE_JSON_BY_CLUSTER[pilot_cluster]
            ).strip()
            response_path.write_text(response_text, encoding="utf-8")

        if response_text is None:
            print(
                f"Cluster {pilot_cluster}: prompt generated; reasoning is still required. "
                f"Save the response to {response_path.relative_to(PROJECT)}"
            )
            master_rows.append(
                {
                    "Cluster ID": pilot_cluster,
                    "Proposed Cell Type": cell_type_annotation,
                    "Annotation Support": "",
                    "Overall Confidence": "",
                    "Dominant Biological Program": "",
                    "Validation Status": "SKIPPED",
                    "_major_uncertainties": "Reasoning response not yet supplied.",
                    "_functional_modules": "",
                }
            )
            execution_rows.append(
                {
                    "Cluster": pilot_cluster,
                    "Schema": "—",
                    "Genes": "—",
                    "Citations": "—",
                    "Safety": "—",
                    "Overall": "SKIPPED",
                }
            )
            continue

        try:
            analysis = json.loads(response_text)
            print(f"Cluster {pilot_cluster}: loaded response from {response_source}.")
        except json.JSONDecodeError as exc:
            response_load_errors.append(
                f"Invalid JSON from {response_source}: line {exc.lineno}, "
                f"column {exc.colno}: {exc.msg}"
            )

        validation_report = validate_analysis(analysis)
        validation_summary = {
            "schema_passed": validation_report["checks"]["schema_passed"],
            "genes_validated": validation_report["checks"]["genes_validated"],
            "citations_validated": validation_report["checks"]["citations_validated"],
            "safety_passed": validation_report["checks"]["safety_passed"],
            "overall_status": "PASS" if validation_report["valid"] else "FAIL",
        }
        validation_path.write_text(
            json.dumps(validation_report, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        validation_summary_path.write_text(
            json.dumps(validation_summary, indent=2), encoding="utf-8"
        )

        if validation_report["valid"]:
            reasoning_path.write_text(
                json.dumps(analysis, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            biological_report_path.write_text(
                build_biological_report(analysis), encoding="utf-8"
            )
            plain_summary_path.write_text(
                "# Plain-Language Summary\n\n"
                + analysis["plain_language_explanation"]
                + "\n",
                encoding="utf-8",
            )
            annotation_support = analysis["annotation_assessment"]["support_level"]
            overall_confidence = analysis["overall_confidence"]
            modules = [
                module["module_name"] for module in analysis["functional_modules"]
            ]
            dominant_program = modules[0] if modules else ""
            uncertainties = "; ".join(
                analysis["weak_or_uncertain_interpretations"]
            )
            print(f"Cluster {pilot_cluster}: validation PASS.")
        else:
            annotation_support = (
                analysis.get("annotation_assessment", {}).get("support_level", "")
                if isinstance(analysis, dict)
                and isinstance(analysis.get("annotation_assessment"), dict)
                else ""
            )
            overall_confidence = (
                analysis.get("overall_confidence", "")
                if isinstance(analysis, dict)
                else ""
            )
            modules = (
                [
                    module.get("module_name", "")
                    for module in analysis.get("functional_modules", [])
                    if isinstance(module, dict)
                ]
                if isinstance(analysis, dict)
                else []
            )
            dominant_program = modules[0] if modules else ""
            uncertainties = (
                "; ".join(analysis.get("weak_or_uncertain_interpretations", []))
                if isinstance(analysis, dict)
                and isinstance(
                    analysis.get("weak_or_uncertain_interpretations"), list
                )
                else "Response failed deterministic validation."
            )
            print(
                f"Cluster {pilot_cluster}: validation FAIL. "
                f"Review {validation_path.relative_to(PROJECT)}"
            )

        master_rows.append(
            {
                "Cluster ID": pilot_cluster,
                "Proposed Cell Type": (
                    analysis.get("proposed_cell_type", cell_type_annotation)
                    if isinstance(analysis, dict)
                    else cell_type_annotation
                ),
                "Annotation Support": annotation_support,
                "Overall Confidence": overall_confidence,
                "Dominant Biological Program": dominant_program,
                "Validation Status": validation_summary["overall_status"],
                "_major_uncertainties": uncertainties,
                "_functional_modules": "; ".join(modules),
            }
        )
        execution_rows.append(
            {
                "Cluster": pilot_cluster,
                "Schema": "PASS" if validation_summary["schema_passed"] else "FAIL",
                "Genes": "PASS" if validation_summary["genes_validated"] else "FAIL",
                "Citations": (
                    "PASS" if validation_summary["citations_validated"] else "FAIL"
                ),
                "Safety": "PASS" if validation_summary["safety_passed"] else "FAIL",
                "Overall": validation_summary["overall_status"],
            }
        )
    except Exception as exc:
        processing_error = f"{type(exc).__name__}: {exc}"
        print(f"Cluster {pilot_cluster}: processing FAIL — {processing_error}")
        failure_report = {
            "valid": False,
            "checks": {
                "schema_passed": False,
                "genes_validated": False,
                "citations_validated": False,
                "safety_passed": False,
            },
            "errors": {
                "schema": [processing_error],
                "genes": [],
                "citations": [],
                "safety": [],
            },
            "warnings": [],
        }
        failure_summary = {
            "schema_passed": False,
            "genes_validated": False,
            "citations_validated": False,
            "safety_passed": False,
            "overall_status": "FAIL",
        }
        validation_path.write_text(
            json.dumps(failure_report, indent=2), encoding="utf-8"
        )
        validation_summary_path.write_text(
            json.dumps(failure_summary, indent=2), encoding="utf-8"
        )
        cell_type = (
            str(
                annotations.loc[
                    annotations["leiden"].eq(pilot_cluster), "cell_type"
                ].iloc[0]
            )
            if annotations["leiden"].eq(pilot_cluster).any()
            else ""
        )
        master_rows.append(
            {
                "Cluster ID": pilot_cluster,
                "Proposed Cell Type": cell_type,
                "Annotation Support": "",
                "Overall Confidence": "",
                "Dominant Biological Program": "",
                "Validation Status": "FAIL",
                "_major_uncertainties": processing_error,
                "_functional_modules": "",
            }
        )
        execution_rows.append(
            {
                "Cluster": pilot_cluster,
                "Schema": "FAIL",
                "Genes": "FAIL",
                "Citations": "FAIL",
                "Safety": "FAIL",
                "Overall": "FAIL",
            }
        )

validation_table = pd.DataFrame(execution_rows).sort_values(
    "Cluster", key=lambda values: values.map(cluster_sort_key)
)
display(validation_table)


Cluster 0: loaded response from results/phase8/cluster_0/cluster_reasoning_response.json.
Cluster 0: validation PASS.
Cluster 1: loaded response from results/phase8/cluster_1/cluster_reasoning_response.json.
Cluster 1: validation PASS.
Cluster 2: loaded response from results/phase8/cluster_2/cluster_reasoning_response.json.


Cluster 2: validation PASS.
Cluster 3: loaded response from results/phase8/cluster_3/cluster_reasoning_response.json.


Cluster 3: validation PASS.


Cluster 4: loaded response from results/phase8/cluster_4/cluster_reasoning_response.json.


Cluster 4: validation PASS.
Cluster 5: loaded response from results/phase8/cluster_5/cluster_reasoning_response.json.
Cluster 5: validation PASS.
Cluster 6: loaded response from results/phase8/cluster_6/cluster_reasoning_response.json.


Cluster 6: validation PASS.
Cluster 7: loaded response from results/phase8/cluster_7/cluster_reasoning_response.json.
Cluster 7: validation PASS.
Cluster 8: loaded response from results/phase8/cluster_8/cluster_reasoning_response.json.
Cluster 8: validation PASS.


,Cluster,Schema,Genes,Citations,Safety,Overall
0,0,PASS,PASS,PASS,PASS,PASS
1,1,PASS,PASS,PASS,PASS,PASS
2,2,PASS,PASS,PASS,PASS,PASS
3,3,PASS,PASS,PASS,PASS,PASS
4,4,PASS,PASS,PASS,PASS,PASS
5,5,PASS,PASS,PASS,PASS,PASS
6,6,PASS,PASS,PASS,PASS,PASS
7,7,PASS,PASS,PASS,PASS,PASS
8,8,PASS,PASS,PASS,PASS,PASS


## 9. Master summaries

The CSV contains the requested machine-readable fields. The Markdown report adds functional-module and uncertainty detail while retaining the same validation status.

In [9]:
master_internal = pd.DataFrame(master_rows).sort_values(
    "Cluster ID", key=lambda values: values.map(cluster_sort_key)
)
summary_columns = [
    "Cluster ID",
    "Proposed Cell Type",
    "Annotation Support",
    "Overall Confidence",
    "Dominant Biological Program",
    "Validation Status",
]
master_summary = master_internal[summary_columns].copy()
summary_csv_path = PHASE8 / "all_clusters_summary.csv"
summary_md_path = PHASE8 / "all_clusters_summary.md"
master_summary.to_csv(summary_csv_path, index=False)

def markdown_table(frame):
    shown = frame.fillna("").astype(str)
    headers = list(shown.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in shown.itertuples(index=False, name=None):
        clean = [
            str(value).replace("|", "\\|").replace("\n", " ")
            for value in row
        ]
        lines.append("| " + " | ".join(clean) + " |")
    return "\n".join(lines)

total_clusters = len(master_internal)
prompts_generated = len(generated_prompt_paths)
responses_found = int(
    sum((PHASE8 / f"cluster_{cluster}" / "cluster_reasoning_response.json").exists()
        for cluster in all_clusters)
)
passes = int(master_internal["Validation Status"].eq("PASS").sum())
failures = int(master_internal["Validation Status"].eq("FAIL").sum())
skipped = int(master_internal["Validation Status"].eq("SKIPPED").sum())

detail_rows = master_internal[
    [
        "Cluster ID",
        "Proposed Cell Type",
        "Annotation Support",
        "Overall Confidence",
        "_functional_modules",
        "_major_uncertainties",
        "Validation Status",
    ]
].rename(
    columns={
        "_functional_modules": "Dominant Functional Modules",
        "_major_uncertainties": "Major Uncertainties",
    }
)

summary_markdown = f"""# Phase 8 — All-Cluster Reasoning Summary

## Execution totals

- **Total clusters:** {total_clusters}
- **Prompts generated:** {prompts_generated}
- **Reasoning responses found:** {responses_found}
- **Validation passes:** {passes}
- **Validation failures:** {failures}
- **Skipped pending reasoning:** {skipped}

## Cluster results

{markdown_table(detail_rows)}
"""
summary_md_path.write_text(summary_markdown, encoding="utf-8")

display(master_summary)
display(Markdown(summary_markdown))
print(f"Saved global CSV: {summary_csv_path.relative_to(PROJECT)}")
print(f"Saved global Markdown: {summary_md_path.relative_to(PROJECT)}")

,Cluster ID,Proposed Cell Type,Annotation Support,Overall Confidence,Dominant Biological Program,Validation Status
0,0,Cytotoxic CD8 T cells,strongly supported,Moderate,Cytotoxic granule and effector-protease module,PASS
1,1,B cells,strongly supported,High,B-cell receptor-associated identity module,PASS
2,2,IL7R+ memory/helper T cells,partially supported,Moderate,T-cell identity and signaling module,PASS
3,3,Classical monocytes,strongly supported,High,Inflammatory classical-monocyte marker module,PASS
4,4,CD16+ non-classical monocytes,partially supported,Moderate,CD16-associated monocyte identity module,PASS
5,5,NK cells,strongly supported,Moderate,Cytotoxic granule effector module,PASS
6,6,Activated/transitional T cells,partially supported,Low,T-cell-associated marker module,PASS
7,7,Naive/resting T cells,partially supported,Moderate,T-cell identity module,PASS
8,8,Platelets,strongly supported,High,Platelet identity and membrane-function module,PASS


# Phase 8 — All-Cluster Reasoning Summary

## Execution totals

- **Total clusters:** 9
- **Prompts generated:** 9
- **Reasoning responses found:** 9
- **Validation passes:** 9
- **Validation failures:** 0
- **Skipped pending reasoning:** 0

## Cluster results

| Cluster ID | Proposed Cell Type | Annotation Support | Overall Confidence | Dominant Functional Modules | Major Uncertainties | Validation Status |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | Cytotoxic CD8 T cells | strongly supported | Moderate | Cytotoxic granule and effector-protease module; CD8 T-cell identity and state module; Immune-cell communication module; Contextual cellular-regulation signal | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |
| 1 | B cells | strongly supported | High | B-cell receptor-associated identity module; B-cell differentiation and contextual state module; HLA-associated expression module | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |
| 2 | IL7R+ memory/helper T cells | partially supported | Moderate | T-cell identity and signaling module; IL7R-associated memory/helper context module; Cellular transport and metabolic context module | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |
| 3 | Classical monocytes | strongly supported | High | Inflammatory classical-monocyte marker module; Myeloid signaling and effector-context module; Protease-regulatory context module | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |
| 4 | CD16+ non-classical monocytes | partially supported | Moderate | CD16-associated monocyte identity module; Broad innate and extracellular-context module; State-associated and unresolved marker module | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |
| 5 | NK cells | strongly supported | Moderate | Cytotoxic granule effector module; Granule release and cytotoxic-state module; Effector protease regulation and cytotoxic-lymphocyte marker module; Immune communication module; Extracellular innate interaction module | UNKNOWN OR UNCERTAIN: the supplied evidence is insufficient to determine whether cluster 5 is exclusively NK cells rather than a closely related cytotoxic T-cell, NKT-cell, or gamma-delta T-cell population.; UNKNOWN OR UNCERTAIN: the supplied evidence is insufficient to determine a unique NK maturation, activation, or recent target-contact state.; UNKNOWN OR UNCERTAIN: the supplied evidence is insufficient to determine a direct immune mechanism for FGFBP2.; UNKNOWN OR UNCERTAIN: the supplied evidence is insufficient to determine a direct NK-cell role for SPON2.; UNKNOWN OR UNCERTAIN: the supplied evidence is insufficient to determine whether CTSW is functionally required in this cluster.; UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance, secretion, degranulation, target-cell killing, or pathway activation. | PASS |
| 6 | Activated/transitional T cells | partially supported | Low | T-cell-associated marker module; Ribosomal-gene expression module; Broad transcript-associated context module | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |
| 7 | Naive/resting T cells | partially supported | Moderate | T-cell identity module; Trafficking and restrained T-cell-state module; Additional cellular-context module | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |
| 8 | Platelets | strongly supported | High | Platelet identity and membrane-function module; Platelet-associated structural and membrane-context module; Additional transcript context module | UNKNOWN OR UNCERTAIN: transcript enrichment does not establish protein abundance or functional activity.; UNKNOWN OR UNCERTAIN: publication findings from other biological settings may not transfer directly to this PBMC3K cluster.; UNKNOWN OR UNCERTAIN: the supplied evidence does not establish that every cell in the cluster has the same state.; The supplied evidence is insufficient to determine one uniquely defined functional state. | PASS |


Saved global CSV: results/phase8/all_clusters_summary.csv
Saved global Markdown: results/phase8/all_clusters_summary.md


## 10. Cluster 5 regression test

The new Cluster 5 prompt, response, validated reasoning, reports, validation files, and manifest must be byte-identical to the previously validated flat artifacts.

In [10]:
cluster5_new = PHASE8 / "cluster_5"
cluster5_regression_pairs = {
    PHASE8 / "cluster_5_reasoning_prompt.md": cluster5_new / "cluster_reasoning_prompt.md",
    PHASE8 / "cluster_5_input_manifest.json": cluster5_new / "input_manifest.json",
    PHASE8 / "cluster_5_reasoning_response.json": cluster5_new / "cluster_reasoning_response.json",
    PHASE8 / "cluster_reasoning.json": cluster5_new / "cluster_reasoning.json",
    PHASE8 / "biological_interpretation_report.md": cluster5_new / "biological_interpretation_report.md",
    PHASE8 / "plain_language_summary.md": cluster5_new / "plain_language_summary.md",
    PHASE8 / "reasoning_validation_report.json": cluster5_new / "reasoning_validation_report.json",
    PHASE8 / "reasoning_validation_summary.json": cluster5_new / "reasoning_validation_summary.json",
}
regression_rows = []
for baseline, refactored in cluster5_regression_pairs.items():
    baseline_exists = baseline.exists()
    refactored_exists = refactored.exists()
    identical = (
        baseline_exists
        and refactored_exists
        and baseline.read_bytes() == refactored.read_bytes()
    )
    regression_rows.append(
        {
            "baseline": str(baseline.relative_to(PROJECT)),
            "refactored": str(refactored.relative_to(PROJECT)),
            "identical": identical,
        }
    )
cluster5_regression = pd.DataFrame(regression_rows)
display(cluster5_regression)
assert cluster5_regression["identical"].all(), (
    "Cluster 5 regression failed; validated behavior changed."
)
print("Cluster 5 regression PASS: all validated artifacts are byte-identical.")

,baseline,refactored,identical
0,results/phase8/cluster_5_reasoning_prompt.md,results/phase8/cluster_5/cluster_reasoning_pro...,True
1,results/phase8/cluster_5_input_manifest.json,results/phase8/cluster_5/input_manifest.json,True
2,results/phase8/cluster_5_reasoning_response.json,results/phase8/cluster_5/cluster_reasoning_res...,True
3,results/phase8/cluster_reasoning.json,results/phase8/cluster_5/cluster_reasoning.json,True
4,results/phase8/biological_interpretation_repor...,results/phase8/cluster_5/biological_interpreta...,True
5,results/phase8/plain_language_summary.md,results/phase8/cluster_5/plain_language_summar...,True
6,results/phase8/reasoning_validation_report.json,results/phase8/cluster_5/reasoning_validation_...,True
7,results/phase8/reasoning_validation_summary.json,results/phase8/cluster_5/reasoning_validation_...,True


Cluster 5 regression PASS: all validated artifacts are byte-identical.


## 11. Manual reasoning workflow and execution summary

Prompts are ready for all clusters. Responses remain a manual ChatGPT/Codex step; no API is called.

In [11]:
print("-" * 50)
print("Reasoning prompts generated for:")
for cluster in all_clusters:
    print(f"Cluster {cluster}")
print("\nNext step:\n")
print("Run each pending prompt through ChatGPT/Codex.")
print("\nSave each response as:\n")
print("results/phase8/cluster_X/cluster_reasoning_response.json")
print("\nThen rerun the notebook.")
print("-" * 50)

print("\nExecution summary")
print(f"- Clusters processed: {len(all_clusters)}")
print(f"- Prompts generated: {prompts_generated}")
print(f"- Responses validated: {passes + failures}")
print(f"- PASS count: {passes}")
print(f"- FAIL count: {failures}")
print(
    "- Skipped clusters: "
    + (
        ", ".join(
            master_internal.loc[
                master_internal["Validation Status"].eq("SKIPPED"), "Cluster ID"
            ].astype(str)
        )
        if skipped
        else "None"
    )
)
print(f"- Output directory: {PHASE8}")

--------------------------------------------------
Reasoning prompts generated for:
Cluster 0
Cluster 1
Cluster 2
Cluster 3
Cluster 4
Cluster 5
Cluster 6
Cluster 7
Cluster 8

Next step:

Run each pending prompt through ChatGPT/Codex.

Save each response as:

results/phase8/cluster_X/cluster_reasoning_response.json

Then rerun the notebook.
--------------------------------------------------

Execution summary
- Clusters processed: 9
- Prompts generated: 9
- Responses validated: 9
- PASS count: 9
- FAIL count: 0
- Skipped clusters: None
- Output directory: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k/results/phase8


## 12. Workflow

```text
Phase 6 + Phase 7 evidence for each cluster
        ↓
Notebook builds one isolated prompt per cluster
        ↓
Prompt copied into ChatGPT/Codex
        ↓
JSON response saved in that cluster's folder
        ↓
Unchanged validators run independently
        ↓
Unchanged reports generated for passing clusters
        ↓
Global Phase 8 summary updated
```